# 📐 Notebook 03: 3D Residual UNet Baseline Training & Low-Data Benchmark
### Classical Supervised 3D Convolutional Baseline (MONAI ResUNet)

This dedicated notebook runs the **3D Residual UNet Baseline**:
1. **Supervised 3D Residual UNet Training (30 Epochs, AMP)**: Symmetric 5-stage encoder-decoder with residual convolutional units and horizontal concatenation skip connections.
2. **Full-Data Held-Out Test Evaluation**: 3D Dice, IoU, 95th Percentile Hausdorff Distance (mm), and latency.
3. **Low-Data Volumetric Label Efficiency**: Evaluates background collapse under restricted training volumes ($1\%$ to $100\%$).
4. **Artifact Export**: Bundles checkpoints and metrics into `unet_outputs.zip`.

> **Estimated Runtime**: ~2.0 - 2.5 hours on NVIDIA Tesla T4 GPU.


## 1. Hardware & CUDA Environment Verification


In [ ]:
!nvidia-smi

import torch

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Please navigate to Notebook Settings -> Accelerator -> GPU T4!"
    )

## 2. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")

## 3. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

# Setup working directory in /kaggle/working
REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/kaggle/working/tumor_segmentation_3d")

# Clean up broken or incomplete clone from previous failed runs
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

thesis_repo = Path("/kaggle/working/thesis_repo")
if thesis_repo.exists() and not (thesis_repo / "src" / "brats_jepa_3d").exists():
    shutil.rmtree(thesis_repo)

# Clone repository if not already present
if not (work_dir / "src" / "brats_jepa_3d").exists():
    if (thesis_repo / "src" / "brats_jepa_3d").exists():
        work_dir = thesis_repo
    elif Path("/kaggle/working/src/brats_jepa_3d").exists():
        work_dir = Path("/kaggle/working")
    else:
        print(f"Cloning codebase from: {REPO_URL} ...")
        !git clone {REPO_URL} {work_dir}

# Verify package was successfully cloned
src_dir = work_dir / "src"
if not (src_dir / "brats_jepa_3d").exists():
    raise RuntimeError(
        "❌ Clone failed! The package 'brats_jepa_3d' was not found on disk.\n"
        "👉 Please ensure 'Internet' is toggled ON in the Kaggle notebook settings (right sidebar)!"
    )

# Change working directory and update sys.path
os.chdir(str(work_dir))
%cd {work_dir}

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Install in editable mode
!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"


## 4. Dataset Discovery & Health Checks
Verifies whether processed `.npz` volumes exist. If absent, automatically invokes `prepare_data_3d.py` with multi-worker parallel resampling (`--num_workers 4`) and compact `float16` storage (~6.7 MB per volume, upcast to FP32 in RAM upon loading).


In [ ]:
import pandas as pd
import torch

from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

print("=== DATASET DISCOVERY ===")
data_dir = get_dataset_dir("brats_gli_3d")
meta_path = get_metadata_path("brats_gli_3d")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

# If preprocessed dataset is not found, check for raw BraTS data and run prepare_data_3d.py
if not meta_path.exists():
    print("\n⚠️ Preprocessed dataset not found. Running 3D preprocessing from raw BraTS data...")
    !python scripts/prepare_data_3d.py --limit 100 --dtype float16 --num_workers 4
    meta_path = get_metadata_path("brats_gli_3d")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())

    ds = BraTS3DDataset(split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    print(
        "❌ ERROR: Please attach 'brats-3d-datasets' or raw BraTS dataset via '+ Add Input' in Kaggle!"
    )


## 5. Supervised 3D Residual UNet Training (30 Epochs, AMP)
Trains the MONAI 3D Residual UNet baseline using combined Soft Dice and Binary Cross-Entropy (BCE) loss.


In [ ]:
!python scripts/train_unet_3d.py \
    --epochs 30 \
    --batch_size 2 \
    --learning_rate 3e-4 \
    --weight_decay 1e-4 \
    --amp

## 6. Held-Out Test Split Evaluation
Evaluates 3D Residual UNet on the held-out test split ($N=271$ volumes).


In [ ]:
!python scripts/evaluate_3d.py \
    --model_type unet \
    --amp

## 7. Low-Data Volumetric Label Efficiency for 3D UNet
Evaluates 3D UNet under restricted annotation budgets ($1\%$ to $100\%$ labels) demonstrating background collapse under extreme class imbalance.


In [ ]:
!python scripts/evaluate_low_data_3d.py \
    --fractions 0.01 0.05 0.10 0.25 0.50 1.00 \
    --epochs 15 \
    --batch_size 2 \
    --amp

## 8. Export & Package Artifacts
Packages UNet checkpoints, logs, and evaluation metrics into `unet_outputs.zip`.


In [ ]:
!mkdir -p /kaggle/working/export_unet
!cp -r outputs/checkpoints/unet* /kaggle/working/export_unet/ 2>/dev/null || true
!cp -r outputs/metrics/*unet* /kaggle/working/export_unet/ 2>/dev/null || true
!cp -r outputs/logs/*unet* /kaggle/working/export_unet/ 2>/dev/null || true

!cd /kaggle/working && zip -r -q unet_outputs.zip export_unet/
print("✓ unet_outputs.zip ready for download!")
!ls -lh /kaggle/working/unet_outputs.zip